# Matrix elements by shells

## Motivation
Up to this point, the functions surrounding GTOs were defined using the pojections. However this is quite inefficient, as when building `overlap([2,0,0],[2,0,0])` we need to compute `overlap([0,0,0],[0,0,0])`, `overlap([1,0,0],[0,0,0])`, and `overlap([2,0,0],[1,0,0])` when these must already have been computed!! Therefore the idea is to compute once and use this result for all the different needed values. 

In order to procede this way we have to change the idea of individual matrix elements to shell matrices. For example, consider the overlap between two $p$ functions. With the previous approach, we had to compute the matrix element as a function of the gtos and the projections. 

When computing a shell, we aim to compute as a function of the gtos, and the result will then be an array of size `l_dim * l_dim`. Therefore the general approach to compute this way is: 

1. Compute all $S$, $T$, $V$, $R$, or $E$ matrices(tensors) up to the maximum angular momentum _once_ (per cartesian coordinate). 
2. Use these to loop over the projections, where the matrix elements associated with $p1$, $p2$ are the product or contraction over i,j,k,l,m,n (and t,u,v in electron repulsion): 
```
loop over p1
    loop over p2
        i,j,k = p1_components
        k,l,m = p2_components
        property[p1,p2] = accumulation[i,k] + (*) accumulation[j,l] + (*) accumulation[k,m]
```

Or similar. That way the resulting matrix (or tensor) can be acessed direcly by index. The only issue is that it obscures a little bit which angular momenta combination yields each matrix element, but that should be possible to be backtraced using the definition of the GTO struct. 

## Electron repulsion integrals

The biggest time save is in the electron repulsion integrals, due to the need of computing multiple times the Hermite polynomial coefficients. Previously we defined the g_abcd() function as: 

```python
def g_abcd():
    # [...]

    Hermite_integral = R_tuv_n(
        alpha, r_PQ, t_max + tau_max, u_max + nu_max, v_max + phi_max, k_hyper
    )

    g_abcd = 0

    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):
                for tau in range(tau_max):
                    for nu in range(nu_max):
                        for phi in range(phi_max):
                            coefficient_1 = E_ab(basis_1, p1, basis_2, p2, t, u, v)
                            coefficient_2 = E_ab(basis_3, p3, basis_4, p4, tau, nu, phi)
                            integral = Hermite_integral[t + tau, u + nu, v + phi, 0]

                            sign = (-1.0) ** (tau + nu + phi)
                            g_abcd += coefficient_1 * coefficient_2 * integral * sign
    # [...]
```

However, this approach presents two inneficiencies: 
1. the Hermite polynomial coefficients are calculated in the innermost loop, where these can be reused, since to compute the highest order ones, the lower order ones will be computed due to them being recursive intermediates. 
2. In a similar way, a single float is returned by E_ab, while the whole tensor could be returned for much less cost. 

Therefore there are two improvements to be made: 
1. Rewrite the Hermite polynomial coefficient function to take a bottoms-up recursive approach. 
2. Rewrite the `g_abcd` function to compute only once _all_ Hermite polynomial coefficients. 

The solutions are: 

In [1]:
from typing import Union
from numpy.typing import NDArray
import math 
import numpy as np 
from py_mods.src.integrals.GTO import GTO, create_normalized_GTO, _S_3D_legacy, _T_3D_legacy
from py_mods.src.integrals.internal.coulomb_utils import _h_ab_Z_legacy
from py_mods.src.integrals.internal.hermite_utils import R_tuv_n, E_bottoms_up
import time

def E_bottoms_up_legacy(
    A: NDArray[np.float32],
    a:float, 
    i_max:int,
    B: NDArray[np.float32], 
    b:float, 
    j_max:int, 
    t_max: int, 
    out: Union[None, NDArray]=None) -> Union[NDArray[np.float32],None]:
    """
    Bottoms-up calculation of Hermite expansion coefficients E_{ij}^t. In the
    same way OS relations can be built upwards instead of recursive calling, 
    the same principle is applied here: build the zero cases and then 
    recursively build the whole E^{AB}_{t} Hermite polynomial coefficients
    tensor. 

    Parameters
    ----------
    A: NDArray[float], of size (3,)
        Cartesian coordinates of centre A. 
    a: float
        Exponent of Gaussian centered at A.
    i_max: int 
        Maximum angular momentum projection of A. 
    B: NDArray[float], of size (3,)
        Cartesian coordinates of centre B. 
    b: float
        Exponent of Gaussian centered at b.
    i_max: int 
        Maximum angular momentum projection of B.
    t_max: int 
        Maximum value t of the Hermite polynomial. t <= i + j
    
    Returns
    -------
    E_ab_t_array: NDArray[float] of size (i_max, j_max, t)
        Complete Hermite polynomial coefficient tensor
    """

    E_ab_t_array = np.zeros((i_max + 1, j_max + 1, t_max + 1), dtype=np.float32)

    X_ab = B - A

    p = a + b
    mu = (a * b) / p
    X_pa = b / p * X_ab
    X_pb = -a / p * X_ab

    #Base case i = j = t = 0 (Helgaker 9.5.8)
    E_ab_t_array[0, 0, 0] = np.exp(-mu * X_ab**2)
    
    # Compute the [i,0,0] entries (Helgaker 9.5.20)
    for i in range(1, i_max+1):       
        E_ab_t_array[i, 0, 0] +=  X_pa * E_ab_t_array[i - 1, 0, 0]
        E_ab_t_array[i, 0, 0] += 1 / (2 * p) * ((i-1)* E_ab_t_array[i - 2, 0,0])

    # Compute the [0,j,0] entries (Helgaker 9.5.21)
    for j in range(1, j_max+1):
        E_ab_t_array[0, j, 0] +=  X_pb * E_ab_t_array[ 0, j - 1,0]
        E_ab_t_array[0, j, 0] += 1 / (2 * p) * ((j-1)* E_ab_t_array[ 0,j - 2,0])

    # Compute the [i,0,t] entries (Helgaker 9.5.18)
    for i in range(1,i_max+1):
        for t in range(1, t_max+1):
            E_ab_t_array[i,0,t] = (2*p)**(-t) * math.comb(i,t)* E_ab_t_array[i-t,0,0]
        
    # Compute the [0,j,t] entries (Helgaker 9.5.19)
    for j in range(1,j_max+1):
        for t in range(1, t_max+1):
            E_ab_t_array[0,j,t] = (2*p)**(-t) * math.comb(j,t)* E_ab_t_array[0,j-t,0]
    
    # Compute the [i, j, 0] entries (Helgaker 9.5.20)
    for i in range(1, i_max+1):
        for j in range(1, j_max+1):
            E_ab_t_array[i, j, 0] = X_pa * E_ab_t_array[i-1, j, 0] 
            E_ab_t_array[i, j, 0] += (1 / (2*p)) * ((i - 1) * E_ab_t_array[i-2, j, 0] + (j) * E_ab_t_array[i-1, j-1, 0])

    # Compute the rest entries (Helgaker 9.5.17)
    for i in range(1, i_max+1):
        for j in range(1,j_max+1):
            for t in range(1, t_max+1): 
                E_ab_t_array[i,j,t] = 1 / (2*p*t) * (i * E_ab_t_array[i-1,j,t-1] + j * E_ab_t_array[i,j-1,t-1])

    
    return E_ab_t_array

Where the `out` is intended for future execution with preallocated memory. Now the g_abcd can be defined in the naive (no symmetry no screening) way as: 

In [2]:
def g_abcd_shell(gto1: GTO, gto2: GTO, gto3: GTO, gto4: GTO, 
    k_hyper: int = 80) -> NDArray[np.float64]:
    """
    Shell-based ERI tensor calculator. The idea here is that instead of 
    working with projections as in the original g_abcd function, this
    function reuses intermediates. The E_AB coefficients are calculated
    once and then read from this, instead of recomputing them in the innermost
    loop. 

    Recall that Gaussian overlap distributions over cartesian coordinates
    can be expanded as a LC of Hermite Gaussians with fixed Hermite polynomial
    coefficients E^{ab}_{t} which are much easier to manipulate. (Helgaker sec 9.5)

    Parameters
    ----------
    gto1, gto2, gto3, gto4 : GTO
        Gaussian primitive basis functions
    
    Returns
    -------
    NDArray[np.float64]
        ERI tensor for the given shells
    """

    a = gto1.exp
    b = gto2.exp
    c = gto3.exp
    d = gto4.exp

    r_A = gto1.R
    r_B = gto2.R
    r_C = gto3.R
    r_D = gto4.R

    p = a + b
    r_P = (a * r_A + b * r_B) / p

    q = c + d
    r_Q = (c * r_C + d * r_D) / q

    r_PQ = r_P - r_Q

    alpha = p * q / (p + q)

    # We compute over + 1 dimension due to the way the recurrent function works. 
    t_max  = u_max  = v_max = gto1.total_L + gto2.total_L + 1
    tau_max = nu_max = phi_max = gto3.total_L + gto4.total_L + 1
    
    # Compute the hermite auxiliary integral
    Hermite_integral = R_tuv_n(
        alpha, r_PQ, t_max + tau_max, u_max + nu_max, v_max + phi_max, k_hyper
    )

    # Compute coefficients of A and B for each cartesian projection
    E_AB_x = E_bottoms_up(r_A[0], a, gto1.total_L, r_B[0], b, gto2.total_L, t_max)
    E_AB_y = E_bottoms_up(r_A[1], a, gto1.total_L, r_B[1], b, gto2.total_L, u_max)
    E_AB_z = E_bottoms_up(r_A[2], a, gto1.total_L, r_B[2], b, gto2.total_L, v_max)

    E_AB_full = np.zeros((gto1.l_dim, gto2.l_dim, t_max, u_max, v_max))

    # Since there are different projections, the hole AB tensor for each projection pair
    # is obtained by contracting over the angular momenta of each projection in each 
    # cartesian coordinate. 
    for p1, proj_1 in enumerate(gto1.l_projections):
        for p2, proj_2 in enumerate(gto2.l_projections):
            i, k, m = proj_1
            j, l, n = proj_2

            # This way, to each projection vector, there is associated an index p1 or p2.
            # Then t, u, v indicate the degree of the hermite polynomial t <= i+j...
            # By contracting this way, the individual angular momenta indices are removed,
            # and only the total Hermite coefficient between two projections with specific
            # t,u,v is stored. 
            E_AB_full[p1, p2, :, :, :] = np.einsum(
                't, u, v -> tuv', 
                E_AB_x[i, j, :t_max], 
                E_AB_y[k, l, :u_max], 
                E_AB_z[m, n, :v_max]
            )

    # This is repeated for C and D
    E_CD_x = E_bottoms_up(r_C[0], c, gto3.total_L, r_D[0], d, gto4.total_L, tau_max)
    E_CD_y = E_bottoms_up(r_C[1], c, gto3.total_L, r_D[1], d, gto4.total_L, nu_max)
    E_CD_z = E_bottoms_up(r_C[2], c, gto3.total_L, r_D[2], d, gto4.total_L, phi_max)

    E_CD_full = np.zeros((gto3.l_dim, gto4.l_dim, tau_max, nu_max, phi_max))
    for p3, proj_3 in enumerate(gto3.l_projections):
        for p4, proj_4 in enumerate(gto4.l_projections):
            ii, kk, mm = proj_3
            jj, ll, nn = proj_4
            E_CD_full[p3, p4, :, :, :] = np.einsum(
                't, u, v -> tuv', 
                E_CD_x[ii, jj, :tau_max], 
                E_CD_y[kk, ll, :nu_max], 
                E_CD_z[mm, nn, :phi_max]
            )
    
    eri_block = np.zeros((gto1.l_dim, gto2.l_dim, gto3.l_dim, gto4.l_dim))

    # Now the eri tensor is defined as (Helgaker 9.9.33): 
    # factor * sum_{t,u,v,tau,nu,phi} E^{AB}_{t,u,v} E^{CD}_{tau,nu,phi} * R_{..}(alpha, R_{PQ})
    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):
                for tau in range(tau_max):
                    for nu in range(nu_max):
                        for phi in range(phi_max):
                            sign = (-1.0) ** (tau + nu + phi)
                            integral = Hermite_integral[t + tau, u + nu, v + phi, 0] * sign
                            eri_block += integral * np.einsum(
                                'ij,kl->ijkl', 
                                E_AB_full[:, :, t, u, v], 
                                E_CD_full[:, :, tau, nu, phi]
                            )
    
    # And the normalization constants are built as a tensor product of the normalization 
    # Constant vectors
    norm_tensor = np.einsum(
                        "i,j,k,l->ijkl", gto1.normalization_constants, gto2.normalization_constants, gto3.normalization_constants, gto4.normalization_constants    
                    )
    
    # And normalization happens as a elementwise product
    eri_block *= norm_tensor

    # And the factor (Helgaker 9.9.33) is applied here 
    return 2 * np.power(np.pi, 2.5) / (p * q * np.sqrt(p + q)) * eri_block

Where it could be possible to use prealocated memory here too. 

In [3]:
from py_mods.src.integrals.Basis import (
    construct_basis_from_lists,
    S_basis_set,
    T_basis_set,
    V_basis_set,
    eri_basis_set,
    eri_basis_set_intermediates,
)

from py_mods.src.integrals.CGTO import create_CGTOClass


def run_benchmark():
    l = 0

    r1 = np.array([0.0, 0.0, 0.0])
    r2 = np.array([0.0, 0.0, 1.4])

    # STO-3G-like exponents for H
    H_exps = np.array([3.42525091, 0.62391373, 0.16885540])
    H_coeffs = np.array([0.15432897, 0.53532814, 0.44463454])

    atom_pos = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.4]])
    atom_charges = np.array([1.0, 1.0])

    print("Constructing basis set (H2 with s, p, functions)...")
    H_1s_1 = create_CGTOClass(r1, H_exps, H_coeffs, l)
    H_1p_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 1)
    H_1d_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 2)
    H_1s_2 = create_CGTOClass(r2, H_exps, H_coeffs, l)
    H_1p_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 1)
    H_1d_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 2)

    bs = construct_basis_from_lists(
        [H_1s_1, H_1p_1, H_1s_2, H_1p_2], atom_pos, atom_charges
    )

    print(f"Total basis functions: {bs.matrix_indices[-1]}")

    print("\n--- Benchmarking ---")

    # Overlap
    t0 = time.time()
    S_bs = S_basis_set(bs)
    t_S = time.time() - t0
    print(f"Overlap (S) time: {t_S:.4f} seconds")

    # Kinetic
    t0 = time.time()
    T_bs = T_basis_set(bs)
    t_T = time.time() - t0
    print(f"Kinetic (T) time: {t_T:.4f} seconds")

    # Nuclear Attraction
    t0 = time.time()
    V_bs = V_basis_set(bs)
    t_V = time.time() - t0
    print(f"Nuclear Attraction (V) time: {t_V:.4f} seconds")

    # Electron Repulsion
    t0 = time.time()
    eri_bs = eri_basis_set_intermediates(bs)
    t_eri = time.time() - t0
    print(f"Electron Repulsion (ERI) time: {t_eri:.4f} seconds")

    print(f"\nTotal integral evaluation time: {t_S + t_T + t_V + t_eri:.4f} seconds")

Where the `eri_basis_set_intermediates` includes the use of `g_abcd` using intermediates.

In [4]:
run_benchmark()

Constructing basis set (H2 with s, p, functions)...
Total basis functions: 8

--- Benchmarking ---
Overlap (S) time: 0.0064 seconds
Kinetic (T) time: 0.0396 seconds
Nuclear Attraction (V) time: 0.1553 seconds
Electron Repulsion (ERI) time: 17.0354 seconds

Total integral evaluation time: 17.2368 seconds


## V matrix elements

The same idea can be applied here. The original nuclear repulsion function was defined as:
```python
def h_ab_Z(
    basis_1,
    projection_1,
    basis_2,
    projection_2,
    charge_atom: float,
    coord_atom,
    k_hyper: int = 80,
) -> float:
    coord_atom = np.array(coord_atom).reshape(-1)

    a = basis_1.exp
    b = basis_2.exp

    r_A = basis_1.R
    r_B = basis_2.R

    i, k, m = projection_1
    j, l, n = projection_2

    t_max = i + j + 1
    u_max = k + l + 1
    v_max = m + n + 1

    p = a + b
    r_P = (a * r_A + b * r_B) / p

    h_ab_total: float = 0.0

    r_PC = r_P - coord_atom
    # print(r_PC)
    R_tuv_n_array = R_tuv_n(p, r_PC, t_max, u_max, v_max, k_hyper)
    charge = charge_atom

    for t in range(t_max + 1):
        for u in range(u_max + 1):
            for v in range(v_max + 1):

                coefficient = E_ab(
                    basis_1, projection_1, basis_2, projection_2, t, u, v
                )

                # print(coefficient)

                hermite_integral: float = R_tuv_n_array[t, u, v, 0]

                # print(f"{t}, {u}, {v}, {0}: {coefficient} {charge} {hermite_integral}")
                # print(f"{coefficient} ")

                h_ab_total += coefficient * hermite_integral

    return -(2 * np.pi / p) * charge_atom * h_ab_total
```

In [5]:
def h_ab_Z_shell(
    gto1,
    gto2,
    charge_atom: float,
    coord_atom,
    k_hyper: int = 80,
) -> float:
    coord_atom = np.array(coord_atom).reshape(-1)

    a = gto1.exp
    b = gto2.exp

    r_A = gto1.R
    r_B = gto2.R

    t_max  = u_max  = v_max = gto1.total_L + gto2.total_L + 1
    p = a + b
    r_P = (a * r_A + b * r_B) / p

    h_ab_total: float = 0.0

    r_PC = r_P - coord_atom
    # Compute the hermite auxiliary integral
    R_tuv_n_array = R_tuv_n(p, r_PC, t_max, u_max, v_max, k_hyper)
    charge = charge_atom

    # Compute coefficients of A and B for each cartesian projection
    E_AB_x = E_bottoms_up(r_A[0], a, gto1.total_L, r_B[0], b, gto2.total_L, t_max)
    E_AB_y = E_bottoms_up(r_A[1], a, gto1.total_L, r_B[1], b, gto2.total_L, u_max)
    E_AB_z = E_bottoms_up(r_A[2], a, gto1.total_L, r_B[2], b, gto2.total_L, v_max)

    E_AB_full = np.zeros((gto1.l_dim, gto2.l_dim, t_max, u_max, v_max))

    # Recalling the definition of the Hermite expansion of the overlap distribution,
    # 
    for p1, proj_1 in enumerate(gto1.l_projections):
        for p2, proj_2 in enumerate(gto2.l_projections):
            i, k, m = proj_1
            j, l, n = proj_2

            # This way, to each projection vector, there is associated an index p1 or p2.
            # Then t, u, v indicate the degree of the hermite polynomial t <= i+j...
            # By contracting this way, the individual angular momenta indices are removed,
            # and only the total Hermite coefficient between two projections with specific
            # t,u,v is stored. 
            E_AB_full[p1, p2, :, :, :] = np.einsum(
                't, u, v -> tuv', 
                E_AB_x[i, j, :t_max], 
                E_AB_y[k, l, :u_max], 
                E_AB_z[m, n, :v_max]
            )

    h_ab_tensor = np.einsum(
        'ijtuv, tuv -> ij', 
        E_AB_full, 
        R_tuv_n_array[:t_max, :u_max, :v_max, 0]
    )

    norm_tensor = np.einsum('i,j->ij', gto1.normalization_constants, gto2.normalization_constants)
    h_ab_tensor *= norm_tensor
    
    return -(2 * np.pi / p) * charge_atom * h_ab_tensor


In [6]:
r1 = np.array([0.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.4])
coord_atom = np.array([0.0, 0.0, 0.7])
charge_atom = 1.0

gto1 = create_normalized_GTO(r1, 3.425, 1)
gto2 = create_normalized_GTO(r2, 0.623, 2)

V_shell = h_ab_Z_shell(gto1, gto2, charge_atom, coord_atom)

V_old = np.zeros((gto1.l_dim, gto2.l_dim))
for p1, proj_1 in enumerate(gto1.l_projections):
    for p2, proj_2 in enumerate(gto2.l_projections):
        unnormalized_val = _h_ab_Z_legacy(gto1, proj_1, gto2, proj_2, charge_atom, coord_atom)
        V_old[p1, p2] = unnormalized_val * gto1.normalization_constants[p1] * gto2.normalization_constants[p2]

print("Same result:", np.allclose(V_shell, V_old))


Same result: True


## Overlap and kinetic energy matrix elements

This shell-based approach can also be performed to compute $S$ and $T$. Consider the original implementation: 

```python
def S_3D_components(    
    basis_1: GTO,
    projection_1: NDArray[int],
    basis_2: GTO,
    projection_2:  NDArray[int],
) -> NDArray[float]:

    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta = basis_2.exp

    overlap_components = np.zeros(3)

    for comp, _ in enumerate(overlap_components):
        overlap_components[comp] = S_1D(
            R_a[comp], R_b[comp], alpha, beta, projection_1[comp], projection_2[comp]
        )

    return overlap_components
```

In [7]:
from py_mods.src.integrals.internal.ST_utils import obara_saika_bottom_up
def S_ab_shell(gto1, gto2) -> NDArray[np.float32]:
    pass

    ax, ay, az = gto1.R
    bx, by, bz = gto2.R

    a = gto1.exp
    b = gto2.exp

    i = gto1.total_L
    j = gto2.total_L

    s_ab_x = obara_saika_bottom_up(ax,bx,a,b,i,j)
    s_ab_y = obara_saika_bottom_up(ay,by,a,b,i,j)
    s_ab_z = obara_saika_bottom_up(az,bz,a,b,i,j)

    S_ab = np.zeros([gto1.l_dim, gto2.l_dim])

    for p1, proj_1 in enumerate(gto1.l_projections):
        for p2, proj_2 in enumerate(gto2.l_projections):
            i,k,m = proj_1
            j,l,n = proj_2
            S_ab[p1, p2] = s_ab_x[i, j] * s_ab_y[k, l] * s_ab_z[m, n]
    
    norm_tensor = np.einsum('i,j->ij', gto1.normalization_constants, gto2.normalization_constants)
    S_ab *= norm_tensor

    return S_ab

And then T:

In [8]:
from py_mods.src.integrals.internal.ST_utils import T_ab_1d_matrix
def T_ab_shell(gto1, gto2) -> NDArray[float]:
    pass

    ax, ay, az = gto1.R
    bx, by, bz = gto2.R

    a = gto1.exp
    b = gto2.exp

    i = gto1.total_L
    j = gto2.total_L

    T_ab_x = T_ab_1d_matrix(ax,bx,a,b,i,j)
    T_ab_y = T_ab_1d_matrix(ay,by,a,b,i,j)
    T_ab_z = T_ab_1d_matrix(az,bz,a,b,i,j)

    s_ab_x = obara_saika_bottom_up(ax,bx,a,b,i,j)
    s_ab_y = obara_saika_bottom_up(ay,by,a,b,i,j)
    s_ab_z = obara_saika_bottom_up(az,bz,a,b,i,j)

    T_ab = np.zeros([gto1.l_dim, gto2.l_dim])

    for p1, proj_1 in enumerate(gto1.l_projections):
        for p2, proj_2 in enumerate(gto2.l_projections):
            i,k,m = proj_1
            j,l,n = proj_2
            T_ab[p1, p2] = (
                T_ab_x[i, j] * s_ab_y[k, l] * s_ab_z[m, n] + 
                s_ab_x[i, j] * T_ab_y[k, l] * s_ab_z[m, n] +
                s_ab_x[i, j] * s_ab_y[k, l] * T_ab_z[m, n]
            )
    
    norm_tensor = np.einsum('i,j->ij', gto1.normalization_constants, gto2.normalization_constants)
    T_ab *= norm_tensor

    return T_ab

In [9]:
def run_primitive_benchmark():
    r1 = np.array([0.0, 0.0, 0.0])
    r2 = np.array([0.0, 0.0, 1.4])
    coord_atom = np.array([0.0, 0.0, 0.7])
    charge_atom = 1.0

    gto1 = create_normalized_GTO(r1, 3.425, 2) 
    gto2 = create_normalized_GTO(r2, 0.623, 2) 
    
    n_iterations = 1000

    print(f"Benchmarking with angular momenta L={gto1.total_L} and L={gto2.total_L}")
    print(f"Number of iterations: {n_iterations}\n")

    # --- Overlap (S) ---
    t0 = time.time()
    for _ in range(n_iterations):
        S_new = S_ab_shell(gto1, gto2)
    t_S_new = time.time() - t0

    t0 = time.time()
    for _ in range(n_iterations):
        S_old = np.zeros((gto1.l_dim, gto2.l_dim))
        for p1, proj_1 in enumerate(gto1.l_projections):
            for p2, proj_2 in enumerate(gto2.l_projections):
                S_old[p1, p2] = _S_3D_legacy(
                    gto1, proj_1, gto1.normalization_constants[p1],
                    gto2, proj_2, gto2.normalization_constants[p2]
                )
    t_S_old = time.time() - t0
    
    print("--- Overlap (S) ---")
    print(f"Match: {np.allclose(S_new, S_old)}")
    print(f"Old: {t_S_old:.4f} s | New: {t_S_new:.4f} s | Speedup: {t_S_old/t_S_new:.2f}x\n")

    # --- Kinetic (T) ---
    t0 = time.time()
    for _ in range(n_iterations):
        T_new = T_ab_shell(gto1, gto2)
    t_T_new = time.time() - t0

    t0 = time.time()
    for _ in range(n_iterations):
        T_old = np.zeros((gto1.l_dim, gto2.l_dim))
        for p1, proj_1 in enumerate(gto1.l_projections):
            for p2, proj_2 in enumerate(gto2.l_projections):
                T_old[p1, p2] = _T_3D_legacy(
                    gto1, proj_1, gto1.normalization_constants[p1],
                    gto2, proj_2, gto2.normalization_constants[p2]
                )
    t_T_old = time.time() - t0

    print("--- Kinetic (T) ---")
    print(f"Match: {np.allclose(T_new, T_old)}")
    print(f"Old: {t_T_old:.4f} s | New: {t_T_new:.4f} s | Speedup: {t_T_old/t_T_new:.2f}x\n")

    # --- Nuclear Attraction (V) ---
    t0 = time.time()
    for _ in range(n_iterations):
        V_new = h_ab_Z_shell(gto1, gto2, charge_atom, coord_atom)
    t_V_new = time.time() - t0

    t0 = time.time()
    for _ in range(n_iterations):
        V_old = np.zeros((gto1.l_dim, gto2.l_dim))
        for p1, proj_1 in enumerate(gto1.l_projections):
            for p2, proj_2 in enumerate(gto2.l_projections):
                unnorm = _h_ab_Z_legacy(gto1, proj_1, gto2, proj_2, charge_atom, coord_atom)
                V_old[p1, p2] = unnorm * gto1.normalization_constants[p1] * gto2.normalization_constants[p2]
    t_V_old = time.time() - t0

    print("--- Nuclear Attraction (V) ---")
    print(f"Match: {np.allclose(V_new, V_old)}")
    print(f"Old: {t_V_old:.4f} s | New: {t_V_new:.4f} s | Speedup: {t_V_old/t_V_new:.2f}x\n")

    # --- Eris ---

    l = 0

    r1 = np.array([0.0, 0.0, 0.0])
    r2 = np.array([0.0, 0.0, 1.4])

    # STO-3G-like exponents for H
    H_exps = np.array([3.42525091, 0.62391373, 0.16885540])
    H_coeffs = np.array([0.15432897, 0.53532814, 0.44463454])

    atom_pos = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.4]])
    atom_charges = np.array([1.0, 1.0])

    print("Constructing basis set (uncontracted (2s,p,d) shell)...")
    H_1s_1 = create_CGTOClass(r1, H_exps, H_coeffs, l)
    H_1p_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 1)
    H_1d_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 2)
    H_1s_2 = create_CGTOClass(r2, H_exps, H_coeffs, l)
    H_1p_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 1)
    H_1d_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 2)

    bs = construct_basis_from_lists(
        [H_1s_1, H_1p_1, H_1s_2, H_1d_2], atom_pos, atom_charges
    )

    t0 = time.time()
    eri_new = eri_basis_set_intermediates(bs)
    t_eri_new = time.time() - t0

    t0 = time.time()
    eri_old = eri_basis_set(bs)
    t_eri_old = time.time() - t0
    print("----- ERI (once) -----")

    print(f"Match: {np.allclose(eri_new, eri_old)}")
    print(f"Old: {t_eri_old:.4f} s | New: {t_eri_new:.4f} s | Speedup: {t_eri_old/t_eri_new:.2f}x\n")

run_primitive_benchmark()


Benchmarking with angular momenta L=2 and L=2
Number of iterations: 1000

--- Overlap (S) ---
Match: True
Old: 0.4460 s | New: 0.0542 s | Speedup: 8.23x

--- Kinetic (T) ---
Match: True
Old: 1.7215 s | New: 0.1275 s | Speedup: 13.50x

--- Nuclear Attraction (V) ---
Match: True
Old: 12.3942 s | New: 1.0260 s | Speedup: 12.08x

Constructing basis set (uncontracted (2s,p,d) shell)...
----- ERI (once) -----
Match: True
Old: 1043.3904 s | New: 57.9305 s | Speedup: 18.01x

